# Project 2, Task 2: Statistical Forecasting Models

Task 2 asks which statistical time-series methods suit the charging-demand problem and how they compare with the
baselines of Task 1. This notebook fits exponential smoothing, seasonal ARIMA, seasonal ARIMA with exogenous inputs
and a decomposition with two seasonal periods to the four hourly series. It scores every model under the Task 1
protocol and checks residuals, stability over time and across levels, the effect of weather, and run time.

The protocol of Task 1 is used unchanged: same series, folds, test period, metrics, primary score and baselines.
Its deliverable says the protocol "must be finalized before proceeding to model development", and the project
constitution makes later tasks consume it as saved. The baseline scores per fold are already known, so a new split
at this point would be chosen after seeing results. Fold 1's 26 weeks of training hours are enough for daily and
weekly cycles, so the folds also suit these models.

The notebook runs from the `project2/` folder with Restart and Run All, reads the Task 1 files in `prepared/` and the
transaction file (for the daily weather only), and writes only `prepared/task2_*` files.

Team: Amir Masnavi (1221579), Inês Oliveira (1231205), Manuela Leite (1200720).

## 1 Setup

The setup fixes the seed and the paths, sets one BLAS thread per process so the parallel fits return the same
numbers on every run, and defines the helpers for timing, figures, the decision log and the learned-operations
ledger. It also records the SHA-256 of every Task 1 file, so Section 11 can show that none of them changed. Task 2
adds one package, statsmodels, which the Task 1 pins do not provide; that is the first entry of the decision log.

In [ ]:
import os

for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_var] = "1"  # one BLAS thread per process, so the parallel fits give the same numbers on every run

import hashlib
import io
import json
import sys
import time
import warnings
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import statsmodels
from joblib import Parallel, delayed
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tools.sm_exceptions import ConvergenceWarning, InterpolationWarning
from statsmodels.tsa.seasonal import MSTL
from statsmodels.tsa.statespace.exponential_smoothing import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import acf, adfuller, kpss, pacf

SEED = 42
np.random.seed(SEED)

DATA = Path(os.environ.get("P2_DATA", "Charging_Data.csv"))
OUT = Path("prepared")
FIG = OUT / "figures_task2"
N_JOBS = min(16, os.cpu_count() or 1)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150

RUN_STARTED = time.perf_counter()
section_times = {}
results = {}


@contextmanager
def timed(section):
    """Add the wall time of the block to section_times[section]."""
    start = time.perf_counter()
    try:
        yield
    finally:
        section_times[section] = round(section_times.get(section, 0.0) + time.perf_counter() - start, 2)


def save_fig(name, fig=None):
    """Save a figure as prepared/figures_task2/<name>.png."""
    FIG.mkdir(parents=True, exist_ok=True)
    (fig or plt.gcf()).savefig(FIG / f"{name}.png", bbox_inches="tight", metadata={"Software": None})


def r(value, digits=4):
    """Round a number for the saved JSON files."""
    return round(float(value), digits)


decisions = []


def add_decision(decision_id, section, decision, evidence):
    """Append one entry to the decision log: the conclusion, then the evidence behind it."""
    if any(d["id"] == decision_id for d in decisions):
        raise ValueError(f"Decision {decision_id} is already in the log")
    decisions.append({"id": decision_id, "section": section, "decision": decision, "evidence": evidence})


ledger = []


def add_ledger(operation, fitted_on, level, fold, value):
    """Record one learned operation with the window it was fitted on."""
    ledger.append({"operation": operation, "fitted_on": fitted_on, "level": level, "fold": fold, "value": value})


def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


TASK1_FILES = sorted(OUT.glob("task1_*")) + sorted((OUT / "figures_task1").glob("*.png"))
START_CHECKSUMS = {str(p.relative_to(OUT)): sha256_of(p) for p in TASK1_FILES}
task1_log = pd.read_csv(OUT / "task1_decision_log.csv", dtype=str)
assert task1_log["id"].iloc[-1] == "D14", "Task 2 decisions continue after D14"

add_decision(
    "D24", "1",
    "statsmodels 0.15.0 is added to requirements.txt and is the only new package of Task 2.",
    "ETS, SARIMA, SARIMAX, MSTL, the ADF and KPSS tests and the Ljung-Box test are not in the Task 1 pins; "
    f"statsmodels {statsmodels.__version__} runs with pandas {pd.__version__} and NumPy {np.__version__}.",
)
print(f"Task 1 files under checksum: {len(START_CHECKSUMS)}; parallel workers: {N_JOBS} of {os.cpu_count()} cores")

## 2 Task 1 inputs and the scoring check

The protocol, the hourly series, the holiday calendar, the folds, the MASE scales and the baseline scores are read
from Task 1 and checked against each other: 15,336 hours, no gap, network equal to the sum of the districts, and fold
dates equal to the protocol. The models work on log(1 + count), built here once for the four levels.

In [ ]:
LEVELS = ["network", "Tongxiang", "Xiuzhou", "Nanhu"]
DISTRICTS = LEVELS[1:]
METHODS = ["ets", "sarima", "sarimax_cal", "sarimax_weather", "mstl_arima"]
FAMILY = {"ets": "ETS", "sarima": "SARIMA", "sarimax_cal": "SARIMAX", "sarimax_weather": "SARIMAX",
          "mstl_arima": "MSTL + ARIMA", "snaive_day": "baseline", "snaive_week": "baseline", "mean_4w": "baseline"}
BASELINES = ["snaive_day", "snaive_week", "mean_4w"]
METHOD_COLORS = dict(zip(METHODS + ["mean_4w"], sns.color_palette("tab10", 6)))
DATE_COLS = ["train_from", "train_before", "validate_from", "validate_before"]

with timed("2 Task 1 inputs and the scoring check"):
    protocol = json.loads((OUT / "task1_protocol.json").read_text())
    series = pd.read_csv(OUT / "task1_hourly_series.csv", parse_dates=["hour"]).set_index("hour")
    assert len(series) == 15_336 and series.index[-1] == pd.Timestamp("2021-09-30 23:00")
    assert not series["is_gap"].any()
    assert (series["network"] == series[DISTRICTS].sum(axis=1)).all()
    assert protocol["problem"]["levels"] == LEVELS and protocol["metrics"]["mase_period"] == 24

    calendar = pd.read_csv(OUT / "task1_calendar.csv", parse_dates=["date"])
    CAL = calendar.set_index("date")
    DAY_TYPE = pd.Series(  # same rule as the Task 1 notebook (Section 2.3)
        np.select([CAL["is_holiday"], CAL["is_adjusted_workday"], CAL["day_of_week"] >= 5],
                  ["holiday", "adjusted working day", "weekend"], default="normal weekday"),
        index=CAL.index, name="day_type",
    )

    FOLDS = pd.read_csv(OUT / "task1_folds.csv", parse_dates=DATE_COLS)
    saved_folds = FOLDS[["fold", *DATE_COLS]].assign(**{c: FOLDS[c].dt.strftime("%Y-%m-%d") for c in DATE_COLS})
    assert saved_folds.to_dict("records") == protocol["folds"]

    task1_ledger = pd.read_csv(OUT / "task1_learned_operations_ledger.csv")
    SCALES = {(row.level, int(row.fold)): float(row.value) for row in task1_ledger.itertuples()}
    assert len(SCALES) == 16

    base_scores = pd.read_csv(OUT / "task1_baseline_scores.csv")
    base_by_period = pd.read_csv(OUT / "task1_baseline_scores_by_period.csv")

    Y_LOG = np.log1p(series[LEVELS].astype(float)).asfreq("h")
    HOURS = Y_LOG.index


def fold_index(fold):
    """Positions in the hourly index: training start, training end (exclusive), validation end, 00:00 issuances."""
    row = FOLDS.set_index("fold").loc[fold]
    start, end, stop = (int(HOURS.searchsorted(row[c])) for c in ("train_from", "train_before", "validate_before"))
    return start, end, stop, np.arange(end, stop, 24)


for fold in FOLDS["fold"]:
    *_, issuances = fold_index(fold)
    assert len(issuances) == FOLDS.set_index("fold").at[fold, "validation_issuances"]
    assert all(HOURS[i].hour == 0 for i in issuances)

display(FOLDS[["fold", *DATE_COLS, "train_issuances", "validation_issuances"]])

The scoring functions copy the Task 1 rules: MAE, RMSE and MASE per method, level, fold and horizon, with the MASE
scale taken from the Task 1 ledger, and a primary score that averages MASE over the 24 horizons, then the four levels,
then the four folds. As a check, the same function applied to the saved baseline scores must return the Task 1
values 0.869, 1.000 and 1.047.

In [ ]:
def score_forecasts(fc):
    """MAE, RMSE and MASE per (method, level, fold, horizon), in the layout of task1_baseline_scores.csv."""
    e = fc.assign(abs_err=(fc["forecast"] - fc["actual"]).abs(), sq_err=(fc["forecast"] - fc["actual"]) ** 2)
    g = (e.groupby(["method", "level", "fold", "horizon"], sort=False)
          .agg(mae=("abs_err", "mean"), mse=("sq_err", "mean")).reset_index())
    g["rmse"] = np.sqrt(g.pop("mse"))
    g["scale"] = [SCALES[(lvl, f)] for lvl, f in zip(g["level"], g["fold"])]
    g["mase"] = g["mae"] / g["scale"]
    return g[["method", "level", "fold", "horizon", "mae", "rmse", "mase", "scale"]]


def score_by_period(fc):
    """The same errors by month and by day type of the target day, as in task1_baseline_scores_by_period.csv."""
    target = fc["issuance"] + pd.to_timedelta(fc["horizon"] - 1, unit="h")
    e = fc.assign(abs_err=(fc["forecast"] - fc["actual"]).abs(), sq_err=(fc["forecast"] - fc["actual"]) ** 2,
                  month=target.dt.strftime("%Y-%m"), day_type=DAY_TYPE.reindex(fc["issuance"]).to_numpy())
    parts = []
    for kind in ["month", "day_type"]:
        p = (e.groupby(["method", "level", "fold", kind], sort=False)
              .agg(n_rows=("abs_err", "size"), mae=("abs_err", "mean"), mse=("sq_err", "mean")).reset_index())
        p = p.rename(columns={kind: "period"})
        p["rmse"] = np.sqrt(p.pop("mse"))
        p["mase"] = p["mae"] / [SCALES[(lvl, f)] for lvl, f in zip(p["level"], p["fold"])]
        p.insert(3, "period_kind", kind)
        parts.append(p)
    return pd.concat(parts, ignore_index=True)[["method", "level", "fold", "period_kind", "period", "n_rows", "mae",
                                                "rmse", "mase"]]


def primary_score(scores, metric="mase"):
    """Mean over horizons, then over levels with equal weight, then over folds. Returns (overall, per fold)."""
    per_level = scores.groupby(["method", "fold", "level"])[metric].mean()
    per_fold = per_level.groupby(["method", "fold"]).mean()
    return per_fold.groupby("method").mean(), per_fold.unstack("fold")


with timed("2 Task 1 inputs and the scoring check"):
    base_primary, base_per_fold = primary_score(base_scores)
    assert base_primary.round(3).to_dict() == {"mean_4w": 0.869, "snaive_day": 1.0, "snaive_week": 1.047}
display(base_per_fold.assign(primary=base_primary))

## 3 Daily weather as a day-D forecast

Task 1 left weather as a candidate that Task 2 should test (D08). The transaction file records one temperature,
humidity and precipitation value per district and day. The file is read with the Task 1 checks: the SHA-256 must match
the protocol, and the 80,081 test rows are dropped right after the start time is parsed. Only temperature is kept,
because it is the only weather variable that moved with daily demand in Task 1. The network value is the mean of the
three districts.

In [ ]:
WEATHER_FROM, WEATHER_TO = pd.Timestamp(protocol["periods"]["train_from"]), pd.Timestamp("2021-09-30")
TEMP = "Temperature(℃)"

with timed("3 Daily weather"):
    raw_bytes = DATA.read_bytes()
    if hashlib.sha256(raw_bytes).hexdigest() != protocol["data"]["sha256"]:
        raise ValueError(f"{DATA} differs from the file Task 1 used; download Charging_Data.csv again from Moodle.")
    raw = pd.read_csv(io.BytesIO(raw_bytes), encoding=protocol["data"]["encoding"], dtype=str, keep_default_na=False,
                      usecols=lambda c: c.strip() in {"District Name", "Start Time", TEMP})
    del raw_bytes
    raw.columns = [c.strip() for c in raw.columns]
    raw = raw.apply(lambda col: col.str.strip())
    start_time = pd.to_datetime(raw["Start Time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
    in_test = start_time >= pd.Timestamp(protocol["periods"]["test_from"])
    assert int(in_test.sum()) == protocol["periods"]["test_rows"]
    raw, start_time = raw[~in_test], start_time[~in_test]  # the test period is dropped right after parsing

    keep = start_time.between(WEATHER_FROM, WEATHER_TO + pd.Timedelta(hours=23, minutes=59, seconds=59))
    keep &= raw["District Name"].isin(DISTRICTS)
    w = pd.DataFrame({"date": start_time[keep].dt.normalize(), "district": raw.loc[keep, "District Name"],
                      "temperature": pd.to_numeric(raw.loc[keep, TEMP], errors="coerce")})
    del raw
    assert w["temperature"].notna().all()
    per_day = w.groupby(["date", "district"])["temperature"]
    assert per_day.nunique().max() == 1, "more than one temperature on a district-day"
    days = pd.date_range(WEATHER_FROM, WEATHER_TO, freq="D")
    WEATHER = per_day.first().unstack("district").reindex(days)[DISTRICTS]
    assert len(WEATHER) == 548 and not WEATHER.isna().any().any()
    WEATHER.insert(0, "network", WEATHER[DISTRICTS].mean(axis=1))
    WEATHER.index.name = "date"
    WEATHER.round(2).to_csv(OUT / "task2_weather_daily.csv", date_format="%Y-%m-%d")

weather_corr = WEATHER[DISTRICTS].corr()
results["weather"] = {"days": len(WEATHER), "min_district_corr": r(weather_corr.to_numpy()[np.triu_indices(3, 1)].min()),
                      "network_mean": r(WEATHER["network"].mean(), 2), "network_min": r(WEATHER["network"].min(), 2),
                      "network_max": r(WEATHER["network"].max(), 2)}
task1_results = json.loads((OUT / "task1_results.json").read_text())
add_decision(
    "D22", "3",
    "Weather enters only through sarimax_weather, as the temperature of day D, read as the forecast available at "
    "00:00 and repeated over the 24 hours; the network value is the mean of the three districts.",
    "Task 1 Spearman correlations with daily network demand: temperature "
    + ", humidity ".join(f"{task1_results['weather_corr'][k]['Spearman']:+.3f}" for k in ("temperature", "humidity"))
    + f", precipitation {task1_results['weather_corr']['precipitation']['Spearman']:+.3f}; one "
    f"temperature per district-day, {len(WEATHER)} days with none missing; the districts correlate at "
    f"{results['weather']['min_district_corr']:.3f} or more.",
)
print("Spearman correlations from Task 1:", {k: v["Spearman"] for k, v in task1_results["weather_corr"].items()})
display(weather_corr.round(4))
display(WEATHER.describe().round(2))

## 4 Modelling choices shared by all models

### 4.1 The log transform

Hourly counts are small and their spread grows with their level: quiet night hours vary by a session or two, busy
afternoons by many more. The data preparation deck lists the log transform for skewed, positive data
[MINDD-03]. With log(1 + count), the zeros are kept and the spread evens out across hours. The table and the figure
compare the standard deviation of each clock hour on the two scales over fold 1's training hours.

In [ ]:
with timed("4 Modelling choices"):
    f1_start, f1_end, _, _ = fold_index(1)
    f1_hours = HOURS[f1_start:f1_end]
    FOLD1_WINDOW = f"{f1_hours[0]:%Y-%m-%d} to {f1_hours[-1]:%Y-%m-%d}"
    spread = []
    sd_raw, sd_log = {}, {}
    for level in LEVELS:
        raw_sd = series.loc[f1_hours, level].groupby(f1_hours.hour).std()
        log_sd = Y_LOG.loc[f1_hours, level].groupby(f1_hours.hour).std()
        sd_raw[level], sd_log[level] = raw_sd, log_sd
        spread.append({"level": level, "raw_sd_min": raw_sd.min(), "raw_sd_max": raw_sd.max(),
                       "raw_ratio": raw_sd.max() / raw_sd.min(), "log_sd_min": log_sd.min(),
                       "log_sd_max": log_sd.max(), "log_ratio": log_sd.max() / log_sd.min()})
    spread = pd.DataFrame(spread).set_index("level")

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for level in LEVELS:
        axes[0].plot(sd_raw[level].index, sd_raw[level].values, marker="o", ms=3, label=level)
        axes[1].plot(sd_log[level].index, sd_log[level].values, marker="o", ms=3, label=level)
    axes[0].set(title="Standard deviation by hour, sessions", xlabel="hour of day", ylabel="sessions per hour")
    axes[1].set(title="Standard deviation by hour, log(1 + sessions)", xlabel="hour of day", ylabel="log scale")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    save_fig("04_1_log_transform", fig)
    plt.show()

display(spread)
results["transform"] = {lvl: {"raw_ratio": r(row.raw_ratio, 2), "log_ratio": r(row.log_ratio, 2)}
                        for lvl, row in spread.iterrows()}


def back(x):
    """Back-transform from log(1 + count): returns the forecast in sessions and whether it was set to zero."""
    value = np.expm1(x)
    return np.maximum(value, 0.0), value < 0


add_decision(
    "D15", "4",
    "Statistical models forecast the 24 hours of day D in one call from their state at 00:00, using observed demand "
    "up to the hour starting at D-1 23:00; the direct strategy with one shared model (D10) stays for the "
    "feature-based models of later tasks.",
    "ETS and ARIMA models produce multi-step forecasts from their own equations; the information set is the one D09 "
    "fixed, and lags of 1 to 23 hours were left out of the catalogue (D11) only because one direct model serves "
    "every horizon.",
)
add_decision(
    "D16", "4",
    "Every model is fitted to log(1 + count); forecasts come back as max(exp(x) - 1, 0) and the cases set to zero "
    "are counted.",
    "On fold 1 training hours the largest hourly standard deviation is "
    + ", ".join(f"{lvl} {row.raw_ratio:.1f}" for lvl, row in spread.iterrows())
    + " times the smallest on the count scale, against "
    + ", ".join(f"{row.log_ratio:.1f}" for _, row in spread.iterrows())
    + " on the log scale; exp(x) - 1 lies between -1 and 0 when x < 0, hence the clip.",
)
add_decision(
    "D17", "4",
    "Coefficients are estimated once per fold and level on the training hours; within the validation quarter the "
    "model is filtered with those fixed coefficients and each 00:00 issuance takes a dynamic 24-hour prediction.",
    "A dynamic prediction from hour e uses observations before e only, so each forecast sees the data known at "
    "issuance; re-estimating every day would mean about 90 fits per fold and level. The perturbation check of "
    "Section 6 confirms that no later observation changes a forecast.",
)

### 4.2 Producing the 24 values and updating every day

ETS and ARIMA models produce the 24 values of day D from their own equations, in one forecast from the state at
00:00. The direct row layout of Task 1 (D10) belongs to feature-based models. The information set stays the one D09
fixed: everything observed up to the hour starting at D-1 23:00. The 1-hour to 23-hour lags that the Task 1 catalogue
excluded were left out only because one direct model serves every horizon, so a statistical model may use them
through its state.

Coefficients are estimated once per fold and level on the training hours. Within the validation quarter, the model
is filtered with those fixed coefficients over the observed hours, and every 00:00 issuance takes a dynamic
prediction of the next 24 hours, which uses observations before that midnight only. Re-estimating every day would
mean about 90 fits per fold and level. In a trial, filtering once gave forecasts identical to appending the data day
by day, about 60 times faster.

The engine below builds each model, fits it on the training hours of one (level, fold) cell and forecasts every
validation issuance. A cell that fails is reported as missing and is never replaced by a baseline. The MSTL model
is decomposed on the training hours only: the daily and weekly seasonal parts of the last training week are repeated
forward, and the ARIMA works on what remains. ETS, ARIMA, SARIMAX and MSTL are not covered in class and were found
through research [Hyndman and Athanasopoulos, fpp3, chapters 8 to 12; Bandara, Hyndman and Bergmeir, 2021].

In [ ]:
BURN_IN = 24 + 168  # residuals dropped at the start of each fit: seasonal differencing and the weekly MSTL period


def calendar_exog(index):
    """Hourly calendar inputs known in advance: day-of-week dummies (Monday is the reference) and the two flags."""
    cal = CAL.reindex(index.normalize())
    assert not cal["day_of_week"].isna().any(), "calendar does not cover every hour"
    X = pd.DataFrame({f"dow_{k}": (cal["day_of_week"].to_numpy() == k).astype(float) for k in range(1, 7)},
                     index=index)
    X["is_holiday"] = cal["is_holiday"].to_numpy().astype(float)
    X["is_adjusted_workday"] = cal["is_adjusted_workday"].to_numpy().astype(float)
    return X


def make_exog(method, level, index):
    if method == "sarimax_cal":
        return calendar_exog(index)
    if method == "sarimax_weather":
        X = calendar_exog(index)
        X["temperature"] = WEATHER[level].reindex(index.normalize()).to_numpy()
        assert not X["temperature"].isna().any()
        return X
    return None


def mstl_seasonal(y_train, n_total):
    """Daily + weekly seasonal part from MSTL on the training hours, then the last week repeated."""
    from statsmodels.tsa.seasonal import MSTL  # imported here so the parallel workers import it the usual way

    seas = MSTL(y_train, periods=(24, 168)).fit().seasonal.sum(axis=1).to_numpy()
    return np.concatenate([seas, np.resize(seas[-168:], n_total - len(seas))])


def make_model(method, structure, y, exog=None):
    from statsmodels.tsa.statespace.exponential_smoothing import ExponentialSmoothing
    from statsmodels.tsa.statespace.sarimax import SARIMAX

    if method == "ets":
        return ExponentialSmoothing(y, trend=structure["trend"], damped_trend=structure["damped"], seasonal=24)
    if method == "mstl_arima":
        return SARIMAX(y, order=structure["order"], trend="c")
    return SARIMAX(y, exog=exog, order=structure["order"], seasonal_order=structure["seasonal_order"])


def run_cell(method, level, fold, structure, forecast=True, y_full=None, params=None):
    """Fit one (method, level, fold) cell on its training hours and forecast every validation issuance.

    With `params` the fit is skipped and the given coefficients are used (perturbation check)."""
    from statsmodels.tools.sm_exceptions import ConvergenceWarning

    start, end, stop, issuances = fold_index(fold)
    if y_full is None:
        y_full = Y_LOG[level].iloc[start:stop]
    n_train = end - start
    exog = make_exog(method, level, y_full.index)
    out = {"method": method, "level": level, "fold": fold, "converged": False, "message": "", "params": None,
           "aic": np.nan, "fit_seconds": np.nan, "forecast_seconds": np.nan}
    try:
        t0 = time.perf_counter()
        y_fit = y_full
        seas = None
        if method == "mstl_arima":
            seas = mstl_seasonal(y_full.iloc[:n_train], len(y_full))
            y_fit = y_full - seas
        if params is None:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                res = make_model(method, structure, y_fit.iloc[:n_train],
                                 None if exog is None else exog.iloc[:n_train]).fit(disp=False, maxiter=500)
            messages = sorted({str(w.message).split("\n")[0] for w in caught
                               if issubclass(w.category, ConvergenceWarning)})
            params = res.params
            if not np.isfinite(params.to_numpy()).all():
                raise ValueError("non-finite coefficients")
            out.update(converged=bool((res.mle_retvals or {}).get("converged", True)), message="; ".join(messages)[:300],
                       aic=float(res.aic), resid=res.resid.to_numpy()[BURN_IN:],
                       resid_index=y_fit.index[:n_train][BURN_IN:])
        else:
            out["converged"] = True
        out["params"] = params
        out["fit_seconds"] = time.perf_counter() - t0
        if forecast:
            t1 = time.perf_counter()
            filtered = make_model(method, structure, y_fit, exog).filter(params)
            fc = np.empty((len(issuances), 24))
            for i, e in enumerate(issuances - start):
                fc[i] = filtered.get_prediction(start=e, end=e + 23, dynamic=True).predicted_mean.to_numpy()
                if seas is not None:
                    fc[i] += seas[e:e + 24]
            if not np.isfinite(fc).all():
                raise ValueError("non-finite forecasts")
            out.update(forecast_log=fc, issuances=HOURS[issuances], forecast_seconds=time.perf_counter() - t1)
    except Exception as exc:  # the cell is reported as missing, never replaced by a baseline
        out.update(converged=False, message=f"{type(exc).__name__}: {exc}"[:300], params=None)
        out.pop("forecast_log", None)
    return out

## 5 Structure selection on fold 1

### 5.1 Differencing and the autocorrelation of the differenced series

Each structure is chosen once, on fold 1's training hours (2020-04-01 to 2020-09-30), and then fixed for the four
folds; only the coefficients change from fold to fold. For SARIMA the seasonal difference at lag 24 follows from the
daily cycle that Task 1 found (D05). Whether a further first difference is needed is read from the ADF test (null:
a unit root) and the KPSS test (null: stationary) on the lag-24 difference; neither is covered in class, and both
were found through research. The ACF and PACF of the lag-24 difference show which short and seasonal lags remain.

In [ ]:
with timed("5 Structure selection on fold 1"):
    stationarity = []
    for level in LEVELS:
        y = Y_LOG[level].iloc[f1_start:f1_end]
        for name, x in [("log series", y), ("lag-24 difference", y.diff(24).dropna())]:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", InterpolationWarning)
                adf_stat, adf_p, *_ = adfuller(x, autolag="AIC")
                kpss_stat, kpss_p, *_ = kpss(x, regression="c", nlags="auto")
            stationarity.append({"level": level, "series": name, "adf_stat": adf_stat, "adf_p": adf_p,
                                 "kpss_stat": kpss_stat, "kpss_p": kpss_p})
    stationarity = pd.DataFrame(stationarity)
    D_ORDER = {}
    for level in LEVELS:
        row = stationarity[(stationarity["level"] == level) & (stationarity["series"] == "lag-24 difference")].iloc[0]
        D_ORDER[level] = 0 if (row.adf_p < 0.05 and row.kpss_p > 0.05) else 1

    fig, axes = plt.subplots(len(LEVELS), 2, figsize=(12, 9), sharex=True)
    lags = np.arange(1, 169)
    bound = 1.96 / np.sqrt(f1_end - f1_start - 24)
    for i, level in enumerate(LEVELS):
        x = Y_LOG[level].iloc[f1_start:f1_end].diff(24).dropna()
        for j, (name, values) in enumerate([("ACF", acf(x, nlags=168)[1:]), ("PACF", pacf(x, nlags=168)[1:])]):
            ax = axes[i, j]
            ax.vlines(lags, 0, values, color=METHOD_COLORS["sarima"], lw=0.8)
            ax.axhspan(-bound, bound, color="0.85")
            for lag in (24, 48, 168):
                ax.axvline(lag, color="0.5", ls=":", lw=0.8)
            ax.set(title=f"{level}: {name} of the lag-24 difference", ylim=(-0.6, 0.8))
    for ax in axes[-1]:
        ax.set_xlabel("lag (hours)")
    fig.tight_layout()
    save_fig("05_2_acf_pacf", fig)
    plt.show()

display(stationarity.set_index(["level", "series"]))
print("non-seasonal differencing d per level:", D_ORDER)
acf_d = {lvl: {str(k): r(acf(Y_LOG[lvl].iloc[f1_start:f1_end].diff(24).dropna(), nlags=168)[k], 3)
               for k in (1, 2, 24, 168)} for lvl in LEVELS}
results["stationarity"] = {"d": D_ORDER, "acf_lag24_difference": acf_d,
                           "table": stationarity.round(4).to_dict("records")}
print(pd.DataFrame(acf_d).T)

### 5.2 Candidate structures and the choice by AIC

A short list of candidates per family is compared by AIC on the same training hours, separately for each level:

- ETS with an additive daily season on the log scale, without trend or with a damped trend (2 candidates);
- SARIMA (p, d, q)(P, 1, Q)24 with p in {1, 2}, q in {0, 1} and (P, Q) in {(0, 1), (1, 1)} (8 candidates);
- an ARIMA(p, 0, q) with a constant on the MSTL-adjusted series, p in {1, 2} and q in {0, 1} (4 candidates).

The two SARIMAX variants keep the SARIMA structure of their level and only add inputs, so the comparison isolates
the inputs. The fits run in parallel and every candidate is saved with its AIC in `task2_structures.csv`.

In [ ]:
def label(method, s):
    if method == "ets":
        return "trend=damped" if s["trend"] else "trend=none"
    if method == "mstl_arima":
        return f"ARIMA{s['order']} + const".replace(" ", "")
    return f"{s['order']}{s['seasonal_order'][:3]}24".replace(" ", "")


def candidates(method, level):
    if method == "ets":
        return [{"trend": False, "damped": False}, {"trend": True, "damped": True}]
    if method == "sarima":
        return [{"order": (p, D_ORDER[level], q), "seasonal_order": (P, 1, Q, 24)}
                for p in (1, 2) for q in (0, 1) for P, Q in ((0, 1), (1, 1))]
    if method == "mstl_arima":
        return [{"order": (p, 0, q)} for p in (1, 2) for q in (0, 1)]
    raise ValueError(method)


with timed("5 Structure selection on fold 1"):
    jobs = [(m, lvl, s) for m in ("sarima", "ets", "mstl_arima") for lvl in LEVELS for s in candidates(m, lvl)]
    grid_out = Parallel(n_jobs=N_JOBS)(delayed(run_cell)(m, lvl, 1, s, forecast=False) for m, lvl, s in jobs)
    grid = pd.DataFrame([{"method": m, "level": lvl, "candidate": label(m, s), "aic": o["aic"],
                          "converged": o["converged"], "message": o["message"], "structure": s}
                         for (m, lvl, s), o in zip(jobs, grid_out)])
    assert grid["converged"].all(), grid.loc[~grid["converged"], ["method", "level", "candidate", "message"]]
    grid["chosen"] = grid["aic"] == grid.groupby(["method", "level"])["aic"].transform("min")
    assert (grid.groupby(["method", "level"])["chosen"].sum() == 1).all()

    STRUCTURES = {(row.method, row.level): row.structure for row in grid[grid["chosen"]].itertuples()}
    inherit = []
    for level in LEVELS:
        for m in ("sarimax_cal", "sarimax_weather"):
            STRUCTURES[(m, level)] = STRUCTURES[("sarima", level)]
            inherit.append({"method": m, "level": level,
                            "candidate": f"inherits sarima {label('sarima', STRUCTURES[('sarima', level)])}",
                            "aic": np.nan, "chosen": True})
    structures = pd.concat([grid[["method", "level", "candidate", "aic", "chosen"]], pd.DataFrame(inherit)],
                           ignore_index=True)
    structures["aic"] = structures["aic"].round(2)
    structures.to_csv(OUT / "task2_structures.csv", index=False)
    assert (structures.groupby(["method", "level"])["chosen"].sum() == 1).all()
    for (m, level), s in STRUCTURES.items():
        if m in ("ets", "sarima", "mstl_arima"):
            add_ledger(f"AIC selection of the {m} structure", FOLD1_WINDOW, level, 1, label(m, s))


def aic_gap(method, level):
    a = grid[(grid["method"] == method) & (grid["level"] == level)]["aic"].sort_values().to_numpy()
    return a[1] - a[0]


chosen_table = pd.DataFrame({m: {lvl: label(m, STRUCTURES[(m, lvl)]) for lvl in LEVELS}
                             for m in ("ets", "sarima", "mstl_arima")})
gaps = pd.DataFrame({m: {lvl: aic_gap(m, lvl) for lvl in LEVELS} for m in ("ets", "sarima", "mstl_arima")})
display(grid.pivot_table(index=["method", "candidate"], columns="level", values="aic").round(1)[LEVELS])
display(chosen_table)
print("AIC gap between the chosen and the next candidate")
display(gaps.round(1))
results["structures"] = {f"{m}|{lvl}": label(m, s) for (m, lvl), s in STRUCTURES.items()}
results["aic_gap"] = {m: {lvl: r(v, 1) for lvl, v in col.items()} for m, col in gaps.items()}

for decision_id, m, name in [("D18", "ets", "ETS"), ("D19", "sarima", "SARIMA"), ("D20", "mstl_arima", "MSTL + ARIMA")]:
    add_decision(
        decision_id, "5",
        f"The {name} structure per level, fixed for the four folds: "
        + "; ".join(f"{lvl} {label(m, STRUCTURES[(m, lvl)])}" for lvl in LEVELS) + ".",
        "Lowest AIC on fold 1 training hours (" + FOLD1_WINDOW + "); gap to the next candidate "
        + ", ".join(f"{lvl} {gaps.at[lvl, m]:.1f}" for lvl in LEVELS)
        + (f"; d from the ADF and KPSS tests on the lag-24 difference: {D_ORDER}" if m == "sarima" else "") + ".",
    )
add_decision(
    "D21", "5",
    "sarimax_cal adds 6 day-of-week dummies (Monday is the reference), the holiday flag and the adjusted-working-day "
    "flag to the SARIMA structure of the level, without a search of its own; sarimax_weather adds the day-D "
    "temperature to these inputs.",
    "The calendar is published in advance (D02); Task 1 found a weekly cycle the daily season does not carry "
    "(D05: snaive_week 1.047) and holiday effects (D06); keeping the SARIMA structure isolates the effect of the inputs.",
)

### 5.3 How each model treats the daily and weekly cycles

## 6 Fitting and forecasting in the four folds

The 80 cells (5 variants, 4 levels, 4 folds) run in parallel, with the slowest ones first. Each forecast is converted
back to sessions with max(exp(x) - 1, 0) and joined with the observed count of its target hour.

In [ ]:
COST = {"sarimax_weather": 5, "sarimax_cal": 5, "sarima": 2, "ets": 1, "mstl_arima": 1}

with timed("6 Fitting and forecasting"):
    jobs = sorted([(m, lvl, f) for m in METHODS for lvl in LEVELS for f in FOLDS["fold"]],
                  key=lambda j: (-COST[j[0]] * j[2], METHODS.index(j[0]), LEVELS.index(j[1])))
    t0 = time.perf_counter()
    outs = Parallel(n_jobs=N_JOBS)(delayed(run_cell)(m, lvl, f, STRUCTURES[(m, lvl)]) for m, lvl, f in jobs)
    fit_wall_seconds = time.perf_counter() - t0
    CELLS = {(o["method"], o["level"], o["fold"]): o for o in outs}

    FITS = pd.DataFrame([{
        "method": m, "level": lvl, "fold": f, "n_params": 0 if o["params"] is None else len(o["params"]),
        "params": "" if o["params"] is None else json.dumps({k: round(float(v), 6) for k, v in o["params"].items()}),
        "converged": o["converged"] and "forecast_log" in o, "message": o["message"],
        "fit_seconds": round(o["fit_seconds"], 2), "forecast_seconds": round(o["forecast_seconds"], 2),
    } for (m, lvl, f), o in sorted(CELLS.items(), key=lambda kv: (METHODS.index(kv[0][0]), LEVELS.index(kv[0][1]),
                                                                    kv[0][2]))])
    rows = []
    for (m, lvl, f), o in CELLS.items():
        if "forecast_log" not in o or not o["converged"]:
            continue
        fc_log = o["forecast_log"]
        value, clipped = back(fc_log)
        targets = o["issuances"].to_numpy()[:, None] + np.arange(24) * np.timedelta64(1, "h")
        rows.append(pd.DataFrame({
            "method": m, "level": lvl, "fold": f, "issuance": np.repeat(o["issuances"], 24),
            "horizon": np.tile(np.arange(1, 25), len(o["issuances"])), "forecast_log": fc_log.ravel(),
            "forecast": value.ravel(), "clipped": clipped.ravel(),
            "actual": series[lvl].reindex(targets.ravel()).to_numpy(),
        }))
    FC = pd.concat(rows, ignore_index=True)
    FC = (FC.assign(_m=FC["method"].map(METHODS.index), _l=FC["level"].map(LEVELS.index))
            .sort_values(["_m", "_l", "fold", "issuance", "horizon"]).drop(columns=["_m", "_l"])
            .reset_index(drop=True))
    assert FC["actual"].notna().all()
    assert FC["issuance"].max() < pd.Timestamp(protocol["periods"]["test_from"])
    n_issuances = FOLDS.set_index("fold")["validation_issuances"]
    for (m, lvl, f), g in FC.groupby(["method", "level", "fold"]):
        assert len(g) == 24 * n_issuances[f], (m, lvl, f)

    for (m, lvl, f), o in CELLS.items():
        if o["params"] is not None:
            start, end, _, _ = fold_index(f)
            add_ledger(f"coefficients of {m}", f"{HOURS[start]:%Y-%m-%d} to {HOURS[end - 1]:%Y-%m-%d}", lvl, f,
                       len(o["params"]))
            if m == "mstl_arima":
                add_ledger("MSTL decomposition, periods 24 and 168",
                           f"{HOURS[start]:%Y-%m-%d} to {HOURS[end - 1]:%Y-%m-%d}", lvl, f, "seasonal part")

missing = FITS[~FITS["converged"]]
print(f"{len(FITS)} cells fitted in {fit_wall_seconds:.0f} s with {N_JOBS} workers; missing cells: {len(missing)}")
display(missing if len(missing) else FITS.groupby("method")[["fit_seconds", "forecast_seconds"]].max())
clip_table = FC.groupby(["method", "level"])["clipped"].sum().unstack("level")[LEVELS].reindex(METHODS)
print("forecasts set to zero by the back-transform")
display(clip_table)

To check that nothing after the issuance reaches a forecast, the network series of fold 2 is changed from
2021-02-01 00:00 on: every later count is multiplied by 10. The model is refiltered with the same coefficients and
the forecast issued at that midnight is computed again. It has to be unchanged for every model.

In [ ]:
def perturbation_check(method, level="network", fold=2, issuance="2021-02-01"):
    """Multiply every count from the issuance on by 10, refilter with the same coefficients, compare the forecast."""
    o = CELLS[(method, level, fold)]
    start, end, stop, _ = fold_index(fold)
    e = HOURS.get_loc(pd.Timestamp(issuance)) - start
    y = Y_LOG[level].iloc[start:stop].copy()
    y.iloc[e:] = np.log1p(np.expm1(y.iloc[e:]) * 10)
    pert = run_cell(method, level, fold, STRUCTURES[(method, level)], y_full=y, params=o["params"])
    i = list(o["issuances"]).index(pd.Timestamp(issuance))
    return float(np.abs(pert["forecast_log"][i] - o["forecast_log"][i]).max())


with timed("6 Fitting and forecasting"):
    leak = {m: perturbation_check(m) for m in METHODS}
    assert all(v <= 1e-10 for v in leak.values()), leak
print("largest change in the 2021-02-01 network forecast after multiplying later counts by 10:", leak)
results["perturbation_max_change"] = {m: float(v) for m, v in leak.items()}

## 7 Residual diagnostics

Residuals are taken on the log scale over each fold's training hours, without the first 192 hours that the seasonal
differencing and the weekly MSTL period need to settle. The Ljung-Box test at lags 24 and 168 asks whether any
autocorrelation is left [Ljung and Box, 1978; fpp3, Section 5.4]; it is not covered in class and was found through
research. With 4,000 to 13,000 hours the test rejects white noise for almost any real series, so the size of the
remaining autocorrelation at 24 and 168 hours matters more than the p-value. Mean residuals on holidays against
normal weekdays, and at night against the day, show where a model is biased. One figure per model (fold 1, four
levels) shows the residuals over time, their ACF up to two weeks and their distribution; two of them are shown here.

In [ ]:
with timed("7 Residual diagnostics"):
    diag = []
    for (m, lvl, f), o in sorted(CELLS.items(), key=lambda kv: (METHODS.index(kv[0][0]), LEVELS.index(kv[0][1]),
                                                                  kv[0][2])):
        if "resid" not in o:
            continue
        resid = pd.Series(o["resid"], index=o["resid_index"])
        lb = acorr_ljungbox(resid, lags=[24, 168], return_df=True)
        for lag, row in lb.iterrows():
            diag.append({"method": m, "level": lvl, "fold": f, "test": "ljung_box", "lag": str(lag),
                         "statistic": row["lb_stat"], "p_value": row["lb_pvalue"]})
        dtype = DAY_TYPE.reindex(resid.index.normalize()).to_numpy()
        hour = resid.index.hour
        groups = {"holiday": dtype == "holiday", "normal weekday": dtype == "normal weekday",
                  "night 00-05": hour < 6, "day 06-23": hour >= 6}
        for name, mask in groups.items():
            diag.append({"method": m, "level": lvl, "fold": f, "test": "group_mean", "lag": name,
                         "statistic": resid[mask].mean(), "p_value": np.nan})
        for k in (24, 168):
            diag.append({"method": m, "level": lvl, "fold": f, "test": "acf", "lag": str(k),
                         "statistic": acf(resid, nlags=168)[k], "p_value": np.nan})
    DIAG = pd.DataFrame(diag)
    DIAG.round(6).to_csv(OUT / "task2_diagnostics.csv", index=False)
    assert len(DIAG[DIAG["test"] == "ljung_box"]) == 2 * FITS["converged"].sum()

    for m in METHODS:
        fig, axes = plt.subplots(len(LEVELS), 3, figsize=(13, 10), gridspec_kw={"width_ratios": [2.2, 1.3, 1]})
        for i, lvl in enumerate(LEVELS):
            o = CELLS[(m, lvl, 1)]
            if "resid" not in o:
                continue
            resid = pd.Series(o["resid"], index=o["resid_index"])
            axes[i, 0].plot(resid.index, resid.values, lw=0.3, color=METHOD_COLORS[m])
            axes[i, 0].set(title=f"{lvl}: residuals (log scale), fold 1 training", ylabel="residual")
            values = acf(resid, nlags=336)[1:]
            axes[i, 1].vlines(np.arange(1, 337), 0, values, color=METHOD_COLORS[m], lw=0.6)
            axes[i, 1].axhspan(-1.96 / np.sqrt(len(resid)), 1.96 / np.sqrt(len(resid)), color="0.85")
            for lag in (24, 168, 336):
                axes[i, 1].axvline(lag, color="0.5", ls=":", lw=0.8)
            axes[i, 1].set(title="residual ACF", ylim=(-0.3, 0.5))
            axes[i, 2].hist(resid, bins=60, density=True, color=METHOD_COLORS[m], alpha=0.6)
            grid_x = np.linspace(resid.min(), resid.max(), 200)
            axes[i, 2].plot(grid_x, stats.norm.pdf(grid_x, resid.mean(), resid.std()), color="0.2", lw=1)
            axes[i, 2].set(title="distribution and normal curve")
        axes[-1, 1].set_xlabel("lag (hours)")
        fig.suptitle(f"Residual diagnostics: {m}", y=1.0)
        fig.tight_layout()
        save_fig(f"07_{m}_residuals", fig)
        plt.close(fig)

lb_table = (DIAG[DIAG["test"] == "ljung_box"].groupby(["method", "lag"])
            .agg(stat_min=("statistic", "min"), stat_max=("statistic", "max"), p_max=("p_value", "max"))
            .reindex(METHODS, level="method"))
acf_table = (DIAG[DIAG["test"] == "acf"].pivot_table(index=["method", "level"], columns="lag", values="statistic",
                                                     aggfunc="mean").reindex(METHODS, level="method")
             .reindex(LEVELS, level="level"))
group_table = (DIAG[DIAG["test"] == "group_mean"].pivot_table(index="method", columns="lag", values="statistic",
                                                              aggfunc="mean").reindex(METHODS))
print("Ljung-Box over the 16 (level, fold) cells")
display(lb_table)
print("residual autocorrelation at lags 24 and 168, mean over the four folds")
display(acf_table.round(3))
print("mean residual by group (log scale), mean over levels and folds")
display(group_table.round(4))
results["ljung_box_p_max"] = {m: r(lb_table.loc[m]["p_max"].max(), 6) for m in METHODS}
results["residual_acf"] = {m: {lvl: {k: r(acf_table.loc[(m, lvl), k], 3) for k in ("24", "168")} for lvl in LEVELS}
                           for m in METHODS}
results["residual_group_mean"] = {m: {g: r(v, 4) for g, v in row.items()} for m, row in group_table.iterrows()}

In [ ]:
from IPython.display import Image
for m in ("sarima", "mstl_arima"):
    display(Image(filename=str(FIG / f"07_{m}_residuals.png"), width=900))

## 8 Comparison with the baselines

The model scores are computed exactly like the baseline scores and put in one table with them. A model is useful only
if it beats the reference baseline, the four-week same-hour mean (primary MASE 0.869). The column
`cells_beating_mean_4w` counts the (level, fold) cells, out of 16, where a model's MASE averaged over horizons is below
the reference.

In [ ]:
with timed("8 Comparison"):
    SCORES = score_forecasts(FC)
    for (m, lvl, f), g in SCORES.groupby(["method", "level", "fold"]):
        assert len(g) == 24 and np.allclose(g["scale"], SCALES[(lvl, f)])
    all_scores = pd.concat([SCORES, base_scores], ignore_index=True)
    primary, per_fold = primary_score(all_scores)
    cell_mase = all_scores.groupby(["method", "level", "fold"])["mase"].mean()
    ref_cells = cell_mase.loc["mean_4w"]
    per_level = all_scores.groupby(["method", "level"])[["mae", "rmse", "mase"]].mean()

    comp = []
    for m in METHODS + BASELINES:
        row = {"method": m, "family": FAMILY[m], "primary_mase": primary[m]}
        row.update({f"mase_fold{f}": per_fold.at[m, f] for f in FOLDS["fold"]})
        row.update({f"mae_{lvl}": per_level.at[(m, lvl), "mae"] for lvl in LEVELS})
        row.update({f"rmse_{lvl}": per_level.at[(m, lvl), "rmse"] for lvl in LEVELS})
        mine = cell_mase.loc[m]
        row["cells_beating_mean_4w"] = int((mine < ref_cells.reindex(mine.index)).sum()) if m != "mean_4w" else 0
        row["clipped_forecasts"] = int(FC.loc[FC["method"] == m, "clipped"].sum()) if m in METHODS else 0
        row["missing_cells"] = int(16 - len(mine))
        row["complete"] = row["missing_cells"] == 0
        comp.append(row)
    COMPARISON = pd.DataFrame(comp).sort_values("primary_mase", ignore_index=True)

    fig, ax = plt.subplots(figsize=(9, 4.2))
    order = COMPARISON["method"].tolist()[::-1]
    ax.barh(order, COMPARISON.set_index("method").loc[order, "primary_mase"],
            color=[METHOD_COLORS.get(m, "0.75") for m in order], alpha=0.85)
    for f, marker in zip(FOLDS["fold"], "os^D"):
        ax.scatter(per_fold.loc[order, f], order, marker=marker, color="0.15", s=18, zorder=3, label=f"fold {f}")
    ax.axvline(primary["mean_4w"], color="tab:red", ls="--", lw=1, label="mean_4w")
    ax.set(xlabel="MASE (m = 24)", title="Primary score with the four per-fold values")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    save_fig("08_1_primary_mase", fig)
    plt.show()


def save_scores():
    """Write the score files; called here and again in Section 11."""
    SCORES.round(6).to_csv(OUT / "task2_scores.csv", index=False)
    COMPARISON.round(4).to_csv(OUT / "task2_comparison.csv", index=False)
    FC.assign(issuance=FC["issuance"].dt.strftime("%Y-%m-%d"), forecast_log=FC["forecast_log"].round(6),
              forecast=FC["forecast"].round(6)).to_csv(
        OUT / "task2_forecasts.csv.gz", index=False, compression={"method": "gzip", "mtime": 0})


save_scores()
show = COMPARISON.set_index("method")
display(show[["family", "primary_mase", *[f"mase_fold{f}" for f in FOLDS["fold"]], "cells_beating_mean_4w",
              "missing_cells"]])
display(show[[f"mae_{lvl}" for lvl in LEVELS] + [f"rmse_{lvl}" for lvl in LEVELS]])
results["primary_score"] = {m: r(v) for m, v in primary.items()}
results["primary_score_per_fold"] = {m: {str(f): r(per_fold.at[m, f]) for f in FOLDS["fold"]} for m in primary.index}
results["mase_per_level"] = {m: {lvl: r(per_level.at[(m, lvl), "mase"]) for lvl in LEVELS} for m in primary.index}
results["mae_per_level"] = {m: {lvl: r(per_level.at[(m, lvl), "mae"], 3) for lvl in LEVELS} for m in primary.index}
results["rmse_per_level"] = {m: {lvl: r(per_level.at[(m, lvl), "rmse"], 3) for lvl in LEVELS} for m in primary.index}
results["cells_beating_mean_4w"] = {m: int(show.at[m, "cells_beating_mean_4w"]) for m in METHODS}
results["clipped_forecasts"] = {m: int(show.at[m, "clipped_forecasts"]) for m in METHODS}
results["missing_cells"] = {m: int(show.at[m, "missing_cells"]) for m in METHODS}

## 9 Stability over folds, horizons, levels, months and day types

An average can hide a model that wins in some quarters and loses in others. The breakdowns below show MASE by horizon
for each level, by fold, by level, by month at network level (the 2021-Q1 quarter contains the Spring Festival), and
by day type.

In [ ]:
with timed("9 Stability"):
    horizon = all_scores.groupby(["method", "level", "horizon"])["mase"].mean()
    fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
    for ax, lvl in zip(axes.ravel(), LEVELS):
        for m in METHODS + ["mean_4w"]:
            h = horizon.loc[(m, lvl)]
            ax.plot(h.index, h.values, lw=1.6 if m == "mean_4w" else 1.1, ls="--" if m == "mean_4w" else "-",
                    color=METHOD_COLORS[m], label=m)
        ax.set(title=lvl, ylabel="MASE (mean of 4 folds)", xticks=range(1, 25, 2))
    for ax in axes[1]:
        ax.set_xlabel("horizon (hours after 00:00)")
    axes[0, 0].legend(fontsize=8)
    fig.tight_layout()
    save_fig("09_1_mase_by_horizon", fig)
    plt.show()

    fold_table = per_fold.loc[METHODS + ["mean_4w"]]
    level_table = all_scores.groupby(["method", "level"])["mase"].mean().unstack("level")[LEVELS].loc[METHODS + ["mean_4w"]]
    BY_PERIOD = score_by_period(FC)
    BY_PERIOD.round(6).to_csv(OUT / "task2_scores_by_period.csv", index=False)
    all_period = pd.concat([BY_PERIOD, base_by_period], ignore_index=True)
    month = (all_period[(all_period["period_kind"] == "month")]
             .groupby(["method", "period"])["mase"].mean().unstack("period"))
    month_net = (all_period[(all_period["period_kind"] == "month") & (all_period["level"] == "network")]
                 .groupby(["method", "period"])["mase"].mean().unstack("period"))
    day_type_table = (all_period[all_period["period_kind"] == "day_type"].groupby(["method", "period"])["mase"]
                      .mean().unstack("period")[["normal weekday", "weekend", "holiday", "adjusted working day"]]
                      .loc[METHODS + ["mean_4w"]])

    fig, ax = plt.subplots(figsize=(12, 4))
    for m in METHODS + ["mean_4w"]:
        ax.plot(month_net.columns, month_net.loc[m], marker="o", ms=3, lw=1.6 if m == "mean_4w" else 1.1,
                ls="--" if m == "mean_4w" else "-", color=METHOD_COLORS[m], label=m)
    sf = list(month_net.columns).index("2021-02")
    ax.axvspan(sf - 0.5, sf + 0.5, color="tab:red", alpha=0.08, label="2021 Spring Festival month")
    ax.set(title="Network MASE by month of the validation quarters", ylabel="MASE")
    ax.legend(fontsize=8, ncol=3)
    fig.tight_layout()
    save_fig("09_2_mase_by_month", fig)
    plt.show()

print("MASE per fold (mean over horizons and levels)")
display(fold_table)
print("MASE per level (mean over horizons and folds)")
display(level_table)
print("MASE by month, mean over levels")
display(month.loc[METHODS + ["mean_4w"]])
print("MASE by day type of the target day, mean over levels and folds")
display(day_type_table)
results["mase_by_day_type"] = {m: {t: r(v) for t, v in row.items()} for m, row in day_type_table.iterrows()}
results["mase_by_month"] = {m: {p: r(v) for p, v in row.items()} for m, row in month.loc[METHODS + ["mean_4w"]].iterrows()}
results["mase_by_horizon"] = {m: {str(h): r(v) for h, v in horizon.loc[m].groupby("horizon").mean().items()}
                              for m in METHODS + ["mean_4w"]}

## 10 Weather effect and computational cost

### 10.1 Weather

`sarimax_weather` differs from `sarimax_cal` only by the day-D temperature. The difference between the two answers
whether weather helps once the calendar is known. The recorded temperature stands in for a forecast made at 00:00,
which is better than any real forecast, so a gain here is an upper bound.

In [ ]:
with timed("10 Weather and cost"):
    d_primary = primary["sarimax_weather"] - primary["sarimax_cal"]
    d_fold = per_fold.loc["sarimax_weather"] - per_fold.loc["sarimax_cal"]
    d_cells = cell_mase.loc["sarimax_weather"] - cell_mase.loc["sarimax_cal"]
    weather_table = d_cells.unstack("fold")
    improved = int((d_cells < 0).sum())
    WEATHER_KEPT = bool(d_primary < 0)

display(pd.DataFrame({"sarimax_cal": per_fold.loc["sarimax_cal"], "sarimax_weather": per_fold.loc["sarimax_weather"],
                      "difference": d_fold}).T.assign(primary=[primary["sarimax_cal"], primary["sarimax_weather"],
                                                              d_primary]))
print("difference in MASE per (level, fold): sarimax_weather minus sarimax_cal")
display(weather_table.loc[LEVELS])
results["weather_effect"] = {"primary_difference": r(d_primary, 5),
                             "fold_difference": {str(f): r(v, 5) for f, v in d_fold.items()},
                             "cells_improved": improved}
results["weather_kept"] = WEATHER_KEPT
add_decision(
    "D23", "10",
    ("Weather is kept for later tasks: " if WEATHER_KEPT else "Weather is dropped for later tasks: ")
    + "the day-D temperature " + ("lowers" if WEATHER_KEPT else "does not lower") + " the primary MASE of the SARIMAX model.",
    f"Primary MASE sarimax_weather {primary['sarimax_weather']:.4f} against sarimax_cal {primary['sarimax_cal']:.4f} "
    f"(difference {d_primary:+.4f}); lower in {improved} of the 16 (level, fold) cells; the gain is an upper bound "
    "because the recorded temperature stands in for a forecast.",
)

### 10.2 Computational cost

Each model is fitted once per quarter and forecasts once a day. The table gives the fitting and forecasting time of
each cell, run on a single core. The forecasting time covers a whole validation quarter.

In [ ]:
with timed("10 Weather and cost"):
    cost = FITS.groupby("method")[["fit_seconds", "forecast_seconds"]].agg(["sum", "max"]).reindex(METHODS)
display(cost)
print(f"Section 6 wall time: {fit_wall_seconds:.0f} s with {N_JOBS} workers")
results["cost_seconds"] = {m: {"fit_sum": r(cost.at[m, ("fit_seconds", "sum")], 1),
                               "fit_max": r(cost.at[m, ("fit_seconds", "max")], 1),
                               "forecast_max": r(cost.at[m, ("forecast_seconds", "max")], 1)} for m in METHODS}
results["fit_wall_seconds"] = r(fit_wall_seconds, 0)

## 11 Saved files and integrity checks

The score files are written again with all five variants, and the remaining files follow. The SHA-256 of every
Task 1 file is compared with the value taken at the start of the run.

In [ ]:
with timed("11 Saved files"):
    save_scores()
    FITS.to_csv(OUT / "task2_fits.csv", index=False)
    END_CHECKSUMS = {str(p.relative_to(OUT)): sha256_of(p) for p in TASK1_FILES}
    checks = {name: {"start": START_CHECKSUMS[name], "end": END_CHECKSUMS[name]} for name in START_CHECKSUMS}
    assert all(v["start"] == v["end"] for v in checks.values()), "a Task 1 file changed during the run"
    (OUT / "task2_task1_checksums.json").write_text(json.dumps(checks, indent=1) + "\n")

    ids = sorted(d["id"] for d in decisions)
    assert ids == [f"D{k}" for k in range(15, 26)], ids
    pd.DataFrame(sorted(decisions, key=lambda d: int(d["id"][1:]))).to_csv(OUT / "task2_decision_log.csv", index=False)
    pd.DataFrame(ledger).to_csv(OUT / "task2_learned_operations_ledger.csv", index=False)
    (OUT / "task2_results.json").write_text(json.dumps(results, indent=1, ensure_ascii=False) + "\n")

    test_from = pd.Timestamp(protocol["periods"]["test_from"])
    assert FC["issuance"].max() < test_from and WEATHER.index.max() < test_from
    months = BY_PERIOD.loc[BY_PERIOD["period_kind"] == "month", "period"]
    assert (pd.to_datetime(months) < test_from).all()

run_info = {
    "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "section_times_seconds": section_times,
    "total_seconds": round(time.perf_counter() - RUN_STARTED, 1),
    "n_jobs": N_JOBS, "cpu_count": os.cpu_count(),
    "python": sys.version.split()[0],
    "packages": {"pandas": pd.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
                 "statsmodels": statsmodels.__version__, "joblib": joblib.__version__,
                 "matplotlib": matplotlib.__version__, "seaborn": sns.__version__},
}
(OUT / "task2_run_info.json").write_text(json.dumps(run_info, indent=1) + "\n")
for p in sorted(OUT.glob("task2_*")):
    print(f"{p.name:42s} {p.stat().st_size:>10,d} bytes")
print("run time so far:", run_info["total_seconds"], "s")

## 12 Synthesis and decision log

In [ ]:
statistical = COMPARISON[COMPARISON["family"] != "baseline"].set_index("method")
best = statistical["primary_mase"].idxmin()
beats = statistical.at[best, "primary_mase"] < primary["mean_4w"]
CARRIED = best if beats else None
results["carried_forward"] = CARRIED
add_decision(
    "D25", "12",
    (f"{best} is carried forward as the statistical reference for Task 3." if beats else
     "No statistical model is carried forward: none beats mean_4w."),
    f"Lowest primary MASE among the statistical models: {best} {statistical.at[best, 'primary_mase']:.4f} against "
    f"mean_4w {primary['mean_4w']:.4f}; below mean_4w in {statistical.at[best, 'cells_beating_mean_4w']} of the 16 "
    f"(level, fold) cells; per fold "
    + ", ".join(f"{per_fold.at[best, f]:.3f}" for f in FOLDS["fold"]) + ".",
)
ids = sorted(d["id"] for d in decisions)
assert ids == [f"D{k}" for k in range(15, 26)], ids
pd.DataFrame(sorted(decisions, key=lambda d: int(d["id"][1:]))).to_csv(OUT / "task2_decision_log.csv", index=False)
(OUT / "task2_results.json").write_text(json.dumps(results, indent=1, ensure_ascii=False) + "\n")
with pd.option_context("display.max_colwidth", 220):
    display(pd.read_csv(OUT / "task2_decision_log.csv", dtype=str))